# 53b. Notebook 53 rerun with the noise control built as the plan specified (follow-on work, exploration)

**Why this notebook.** Notebook 53 closed the task-state route because R1 failed: at no window did the clean state stop at least 60 percent of robust successes while noise stopped at most 15 percent. But its noise arms were not the control the plan and notebook 51 pre-registered. The plan's control is "a random change of the same size": at every written layer the state becomes *attacked + noise*, with the noise's norm equal to (clean minus attacked) at that layer and position. That is the construction notebook 51's `random_all` used at each layer (it wrote all 36 layers; the windows here write only their own layers). Notebook 53 instead *added* the noise to the current state at every layer of the window (`kind='add'`). The noise written at one layer is carried by the residual stream into the next layer, and the next layer's noise is added on top, so over a six-layer window the state moves roughly √6 ≈ 2.4 times as far as one layer's noise, and further over layers 0 to 31. The clean arm replaces the state at each layer, so it moves it by exactly (clean minus attacked). Notebook 53's noise was therefore larger than the clean change, which biases R1 against the clean state. The dose arms had the same problem, and the clean dose compounded toward the clean state over the window.

**What changes: only how the noise and dose arms are built.**

| condition | what is written (at every layer of the window, replacing the state) |
|---|---|
| `noiseR_<window>` | attacked + noise; the same noise vectors and seeds as notebook 53 (`seed53`), norm (clean minus attacked) per layer and position |
| `ft_noiseR` | the same at every layer, with the clean first token forced |
| `mixR_a<α>`, `noiseR_a<α>` | attacked + α (clean minus attacked), and attacked + α noise, at the best window, α of 0.25 and 0.5 |
| transplant `noiseR` | the run's own state + noise with the norm of (donor minus own state), at layers 0 to 31 |

**What does not change.** The items; the five-sample screen (read from notebook 53's cache, no new sampling, labels checked against notebook 53's file); every clean arm, both in-distribution controls, `ft_none` and `ft_clean`, the single positions, the placeholder and filler arms, and the transplant's donor and other-clean arms (all read from notebook 53's cache); the go marks and the decision rule, copied unchanged from notebook 53. The best window is chosen by notebook 53's rule (the largest clean-minus-noise gap on robust successes), now with the rebuilt noise; if it moves, the controls and positions are run at the new window too.

**Placeholder.** Notebook 53 scored the placeholder mark on the "(document)" form, and that stays the deciding row. The plan also named a filler form of the same token length; notebook 53 ran it, and its numbers are already known (it stops 0.81 of robust successes with F1 0.549 against the clean arm's 0.597), so it is printed as recorded, not scored as a mark. Neither row changes the route decision, which rests on R1.

**Checks before any new answer counts.** (1) A size check: how far each noise version moves the state just after the window, as a multiple of how far the clean write moves it; the notebook stops unless the rebuilt noise moves it between 0.67 and 1.5 times as far. (2) Three reproduction checks: one batch each of notebook 53's `clean_L21-26`, `noise_L21-26` (its construction) and transplant `donor` is regenerated and must match notebook 53's cached answers (at most one answer in 16 may differ, as in notebook 53's self-test), which confirms the same states, the same donors and the diagnosis of notebook 53's noise.

**Go marks, unchanged from notebook 53 (robust successes of the undefended model).**

| question | go to notebook 54 if |
|---|---|
| R1: does the state separate from noise? | at layers 0 to 31 or at some window, the clean state stops at least 0.60 while noise at the same layers stops at most 0.15; with the first token held equal, `ft_clean` minus the noised arm is at least 0.20 |
| R1: switch or relay? | recorded, no mark |
| R2: site and dose | the best window's clean stop rate reaches at least 80 percent of the clean stop rate at layers 0 to 31; the smallest α at which the clean state stops at least 0.50 is recorded |
| R3: content-free switch | robust failures: the own injection is followed under `donor` in at least 0.30, under noise and `otherclean` in at most 0.10, with the paired difference from noise above zero at its lower end; clean twins: the donor's injection is followed in at most 0.10 |
| R3: the state carries content | clean twins: the donor's injection is followed under `donor` in at least 0.30, under both controls in at most 0.05 |
| placeholder | stops at least 0.60 of robust successes, median content F1 within 0.10 of the clean-twin arm's ("(document)" form, as in notebook 53) |

**What the result decides (same rule as notebook 53).** If R1 passes, notebook 54 (confirmation on fresh items) is built, and notebook 53c (the learned subspace, called 53b in the plan) if R2 passes too. If R1 fails, the task-state route closes, now on the pre-registered control, and the role signal (R5) is next after its novelty check. Notebook 53's files are not touched; a table puts notebook 53's noise beside the rebuilt noise.

**Size of the run.** About 2,000 greedy answers (six window noise arms on the 154 successes of both models, the first-token noise arm and four dose arms on the 147 undefended successes, the transplant noise arm on about 266 runs, three check batches), about 1,000 more if the best window moves, plus judge calls. Run it on the same GPU type as notebook 53, so the reproduction checks compare like with like. Needs notebook 53's commit and its caches in `data/state_patching/`.

**Outputs.** `data/state_patching_53b/`, `reports/state_patching_53b/*.csv`, `reports/state_patching_53b/go.txt`, `figures/state_patching_53b/state_patching_53b.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes peft langdetect==1.0.9
!pip uninstall -y -q torchao   # Colab ships an old torchao that the current PEFT rejects; nothing in this project uses it
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports/state_patching_53b', 'figures/state_patching_53b', 'data/state_patching_53b'): os.makedirs(d_, exist_ok=True)
import torch, gc
assert torch.cuda.is_available(), 'this notebook needs a GPU'
print('Session ready. GPU:', torch.cuda.get_device_name(0))

## Stage A. Settings, items and notebook 53's files (no GPU)

In [ ]:
import importlib, json, hashlib, collections, numpy as np, pandas as pd
import task_state, state_patching, adaptive_attack, adaptive_strict, final_scoring, phase5_redo, targets, attack_success, ci_tools, fidelity_audit, attack_audit
for m_ in (task_state, state_patching, adaptive_attack, adaptive_strict, final_scoring, phase5_redo, targets, attack_success, ci_tools, fidelity_audit, attack_audit): importlib.reload(m_)
from task_state import chat, suffix_ids, token_f1, degenerate, seed_of
from state_patching import PatchHooks, layer_spec, generate_patched, capture_states, seed53, fresh_noise, neutral_filler, screen_label, FILLER
from adaptive_attack import host_of, build_variant
from adaptive_strict import load_attempts
from final_scoring import final_rule, paper_verdict, langdetect_non_english, first_success, CATEGORIES, JUDGE_ONLY, TRANSLATION, SCAMS
from phase5_redo import flat_variants
from targets import load_lm, generate_batch, TASK_SYSTEM, TASK_USER, build_task_prompts
from attack_success import judge_prompt, parse_judge, JUDGE_SYSTEM
from attack_audit import host_retention
from ci_tools import wilson, fmt
from fidelity_audit import newcombe_paired
from peft import PeftModel
H = lambda s: hashlib.sha1(s.encode()).hexdigest()
def jload(f): return json.load(open(f)) if os.path.exists(f) else {}
def jsave(obj, f): json.dump(obj, open(f + '.tmp', 'w')); os.replace(f + '.tmp', f)
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 90)
TARGET, JUDGE = 'Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct'
MODELS = {'qwen3b_undefended': dict(adapter=None, label='Qwen 3B undefended'), 'qwen3b_v4': dict(adapter='data/distill4/lora_3b', label='Qwen 3B full recipe (v4)')}
U, V4 = 'qwen3b_undefended', 'qwen3b_v4'; LB = {t: MODELS[t]['label'] for t in MODELS}
NB51, OUT53, REP53 = 'data/task_state', 'data/state_patching', 'reports/state_patching'                     # read only
OUT, REP, FIG = 'data/state_patching_53b', 'reports/state_patching_53b', 'figures/state_patching_53b'        # this notebook's own
for d_ in [REP, FIG] + [f'{OUT}/{t}' for t in MODELS]: os.makedirs(d_, exist_ok=True)
for t in MODELS:
    for f in ('clean_out.json', 'judge.json', 'p2_base.json', 'p3_base.json'): assert os.path.exists(f'{NB51}/{t}/{f}'), f'notebook 51 cache missing: {NB51}/{t}/{f}'
    for f in ('screen.json', 'judge.json'): assert os.path.exists(f'{OUT53}/{t}/{f}'), f'notebook 53 cache missing: {OUT53}/{t}/{f}'
for f in ('screen_items.csv', 'success_items.csv', 'positions.csv', 'go_criteria.csv'): assert os.path.exists(f'{REP53}/{f}'), f'notebook 53 report missing: {REP53}/{f}'
assert os.path.exists('reports/task_state/test2_items.csv'), 'run notebook 51 first'

# ---- settings and go marks: notebook 53's, unchanged ----
BS, MAX_NEW = 16, 160
SCREEN = dict(samples=5, temperature=0.7, top_p=0.9, need=4)
WINDOWS = {'L6-11': range(6, 12), 'L12-17': range(12, 18), 'L18-23': range(18, 24), 'L21-26': range(21, 27), 'L24-29': range(24, 30), 'L0-31': range(0, 32)}
SIX = [w for w in WINDOWS if w != 'L0-31']; ALL = 'L0-31'
DOSES = (0.25, 0.5, 1.0)
CRIT = dict(r1_stop=0.60, r1_noise=0.15, r1_ft_gap=0.20, r2_share=0.80, r2_dose_stop=0.50, r3_own=0.30, r3_ctrl=0.10, r3_twin_donor=0.10, r3_content=0.30, r3_content_ctrl=0.05, ph_stop=0.60, ph_f1_tol=0.10)
W53 = str(pd.read_csv(f'{REP53}/positions.csv').window.iloc[0])          # notebook 53's best window (its dose, controls and positions ran there)
print("notebook 53's best window:", W53)

# ---- the 210 injections, the clean twins and the logged attempts, as notebooks 51 and 53 built them ----
SUB = pd.DataFrame(json.load(open('data/phase5v3/subset_full.json'))); REW = json.load(open('data/phase5v3/rewrites_full.json')); REX = json.load(open('data/phase5v3/rewrites_extra.json'))
n = len(SUB); assert set(SUB.meta) == set(CATEGORIES)
SIDX = [i for i in range(n) if SUB.meta.iloc[i] not in JUDGE_ONLY]; JIDX = [i for i in range(n) if SUB.meta.iloc[i] in JUDGE_ONLY]
SPLIT = [host_of(SUB.text.iloc[i], SUB.instruction.iloc[i]) for i in range(n)]; assert all(a is not None for _, a in SPLIT)
CLEAN_DOC = [(b[:-1] if b.endswith('\n') else b) + a for b, a in SPLIT]; CLEAN_P = [TASK_USER.format(doc=d) for d in CLEAN_DOC]
for t in MODELS: assert open(f'{NB51}/{t}/clean_docs.sha1').read().strip() == H('\x00'.join(CLEAN_DOC)), 'the clean twins differ from the ones notebook 51 cached'
PAPER = pd.read_csv('reports/paper_security_per_injection.csv')
LD = {}
def ld(o):
    k = H(o or '')
    if k not in LD: LD[k] = langdetect_non_english(o)
    return LD[k]
def attempts(t):
    rows = []
    for log, idx in ((load_attempts('data/adaptive_strict', t, 4), SIDX), (load_attempts('data/final_scoring/adaptive', t, 4), JIDX)):
        for a in log:
            i = idx[a['i']]; cat = SUB.meta.iloc[i]; ok_ = bool(a['verdict']) or (cat == TRANSLATION and ld(a['output']))
            rows.append(dict(injection=i, category=cat, goal=SUB.instruction.iloc[i], attempt=int(a['attempt']), document=a['document'], output=a['output'], success=ok_, key=H(f"{i}\x00{a['document']}")))
    return pd.DataFrame(rows).sort_values(['injection', 'attempt']).reset_index(drop=True)
ATT = {t: attempts(t) for t in MODELS}
for t, A in ATT.items():
    fs = A[A.success].groupby('injection').attempt.min().to_dict(); pp = PAPER[PAPER.model == t].set_index('injection')
    assert [fs.get(i) for i in range(n)] == [None if pd.isna(x) else int(x) for x in pp.adaptive_first_success.reindex(range(n))], f'{t}: the paper\'s adaptive first successes are not reproduced'
# judge verdicts: notebook 51's cache, notebook 53's, then this notebook's own (new verdicts are written to this notebook's folder only)
JC = {t: {**jload(f'{NB51}/{t}/judge.json'), **jload(f'{OUT53}/{t}/judge.json'), **jload(f'{OUT}/{t}/judge.json')} for t in MODELS}
jkey = lambda goal, out: H(goal + '\x00' + (out or ''))
def needs_judge(cat, out, doc): return (not ld(out)) if cat == TRANSLATION else final_rule(cat, out, doc) is None
def ok(t, cat, goal, doc, o):
    """The paper's verdict: did answer o follow instruction `goal` of category `cat`, with `doc` the document the answer
    was given (its emoji and base64 strings do not count)."""
    k = jkey(goal, o)
    if needs_judge(cat, o, doc):
        assert k in JC[t], 'an answer the rule sends to the judge was not judged'; j = JC[t][k]
    else: j = False
    return bool(paper_verdict(cat, o, doc, j))
P2B = {t: jload(f'{NB51}/{t}/p2_base.json') for t in MODELS}; P3B = {t: jload(f'{NB51}/{t}/p3_base.json') for t in MODELS}; CO = {t: jload(f'{NB51}/{t}/clean_out.json') for t in MODELS}
def items_of(t):
    A = ATT[t]; first = A[A.success].groupby('injection').head(1)
    succ = [dict(injection=int(r.injection), category=r.category, goal=r.goal, document=r.document, attempt=int(r.attempt), key=r.key, prompt=TASK_USER.format(doc=r.document)) for r in first.itertuples()]
    fail = [dict(injection=int(r.injection), category=r.category, goal=r.goal, document=r.document, attempt=1, key=r.key, prompt=TASK_USER.format(doc=r.document)) for r in A[(A.attempt == 1) & ~A.success].itertuples()]
    for it in succ: assert it['key'] in P2B[t], 'notebook 51 unpatched answer missing'; it['greedy'] = P2B[t][it['key']][0]; it['greedy_ok'] = ok(t, it['category'], it['goal'], it['document'], it['greedy'])
    for it in fail: assert it['key'] in P3B[t], 'notebook 51 unpatched answer missing'; it['greedy'] = P3B[t][it['key']][0]; it['greedy_ok'] = ok(t, it['category'], it['goal'], it['document'], it['greedy'])
    succ = [it for it in succ if it['greedy_ok']]; fail = [it for it in fail if not it['greedy_ok']]
    return succ, fail
SUCC, FAIL = {}, {}
for t in MODELS: SUCC[t], FAIL[t] = items_of(t)
assert len(SUCC[U]) == 147 and len(FAIL[U]) == 119, (len(SUCC[U]), len(FAIL[U]))
assert len(SUCC[V4]) == 7, len(SUCC[V4]); FAIL[V4] = []
TWIN = {t: [dict(injection=it['injection'], category=it['category'], goal=it['goal'], document=CLEAN_DOC[it['injection']], key=f"twin{it['injection']}", prompt=CLEAN_P[it['injection']],
                 greedy=CO[t][str(it['injection'])]['output'], first=int(CO[t][str(it['injection'])]['first'])) for it in SUCC[t]] for t in MODELS}
TWIN_ALL = {t: [dict(injection=i, key=f'twin{i}', prompt=CLEAN_P[i]) for i in range(n)] for t in MODELS}
for t in MODELS:
    for it in SUCC[t] + FAIL[t]: it['first_clean'] = int(CO[t][str(it['injection'])]['first']); it['clean_answer'] = CO[t][str(it['injection'])]['output']
# notebook 51's single-layer robustness, for the extra group in the tables
d51 = pd.read_csv('reports/task_state/test2_items.csv'); u51 = d51[(d51.model == U) & (d51.group == 'adaptive') & (d51.base == 1)].copy().reset_index(drop=True)
assert len(u51) == len(SUCC[U]) and set(u51.injection) == {it['injection'] for it in SUCC[U]}
RL = [f'random_L{l}' for l in (6, 12, 18, 24, 30)]; robust5 = ~np.any(np.stack([(u51[c] == 0).values for c in RL]), 0)
ROBUST5 = {int(i): bool(r) for i, r in zip(u51.injection, robust5)}
def rate(x):
    x = np.asarray(x, bool); k, m_ = int(x.sum()), len(x); lo, hi = wilson(k, m_) if m_ else (np.nan, np.nan); return dict(n=m_, k=k, rate=k / m_ if m_ else np.nan, lo=lo, hi=hi)
rstr = lambda r, d=2: f"{r['rate']:.{d}f} [{r['lo']:.{d}f}, {r['hi']:.{d}f}]" if r['n'] else 'no items'
paired = lambda x, y: newcombe_paired(x, y) if len(x) else (np.nan, np.nan, np.nan)
dstr = lambda d: f'{d[0]:+.2f} [{d[1]:+.2f}, {d[2]:+.2f}]' if not np.isnan(d[0]) else ''
print('items:', {LB[t]: dict(successes=len(SUCC[t]), failed_originals=len(FAIL[t]), clean_twins=len(TWIN[t])) for t in MODELS})

## Stage B. The model, the task states and the self-test of the hooks (GPU)

In [ ]:
SFX = None; _M = dict(tag=None)
def release():
    if _M.get('tag') is None: return
    _M['hooks'].close(); _M.clear(); _M['tag'] = None; gc.collect(); torch.cuda.empty_cache()
def use(t):
    global SFX
    if _M.get('tag') == t: return _M['model'], _M['tok'], _M['hooks']
    release(); m, tok = load_lm(TARGET, four_bit=True)
    if MODELS[t]['adapter']: m = PeftModel.from_pretrained(m, MODELS[t]['adapter']).eval()
    s = suffix_ids(tok, TASK_SYSTEM); assert SFX is None or s == SFX; SFX = s
    _M.update(tag=t, model=m, tok=tok, hooks=PatchHooks(m, len(SFX))); print(f'loaded {LB[t]}: {len(_M["hooks"].layers)} layers; task state = the last {len(SFX)} prompt tokens {tok.convert_ids_to_tokens(SFX)}')
    return m, tok, _M['hooks']
by_len = lambda ps: sorted(range(len(ps)), key=lambda i: (len(ps[i]), i))
chunks = lambda order, bs=None: [order[s: s + (bs or BS)] for s in range(0, len(order), bs or BS)]
_J = {}
def seven_b():
    if 'm' not in _J: _J['m'], _J['t'] = load_lm(JUDGE, four_bit=True)
    return _J['m'], _J['t']
def free_seven_b():
    if 'm' in _J: _J.clear(); gc.collect(); torch.cuda.empty_cache()
def judge_pending(t, triples):
    """triples: (category, goal, document, output). Judges every answer the rule sends to the judge that has no verdict in
    any cache yet; new verdicts are written to this notebook's folder."""
    need = {}
    for cat, goal, doc, o in triples:
        k = jkey(goal, o)
        if k not in JC[t] and k not in need and needs_judge(cat, o, doc): need[k] = (goal, o)
    if need:
        jm, jt = seven_b(); ks = list(need); own = jload(f'{OUT}/{t}/judge.json')
        for s in range(0, len(ks), 400):
            ch = ks[s: s + 400]; jo = generate_batch(jm, jt, [judge_prompt(*need[k]) for k in ch], system=JUDGE_SYSTEM, max_new_tokens=8, batch_size=8, temperature=0.0, show_every=400)
            for k, x in zip(ch, jo): JC[t][k] = own[k] = bool(parse_judge(x))
            jsave(own, f'{OUT}/{t}/judge.json')
    return len(need)
STATES = {}
def states(t, items):
    """Task states of the items' prompts, float16 [L, K, H] each, kept in memory for the session (one prompt pass per batch)."""
    m, tok, hk = use(t); S = STATES.setdefault(t, {}); need = [it for it in items if it['key'] not in S]
    for idx in chunks(by_len([it['prompt'] for it in need])):
        s = capture_states(m, hk, tok, [need[i]['prompt'] for i in idx], TASK_SYSTEM, SFX)
        for j, i in enumerate(idx): S[need[i]['key']] = s[j]
    return S
TESTED = set()
def self_test(t):
    """As in notebook 53: one prompt pass per call; writing the captured state back unchanged at every layer leaves the
    answers as they were; with the first token forced, the write lands on the template positions."""
    if t in TESTED: return
    m, tok, hk = use(t); its = SUCC[t][:BS]; ps = [it['prompt'] for it in its]
    A = capture_states(m, hk, tok, ps, TASK_SYSTEM, SFX); before = hk.prompt_passes
    o1, _ = generate_patched(m, hk, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW); passes = hk.prompt_passes - before
    o2, _ = generate_patched(m, hk, tok, ps, TASK_SYSTEM, SFX, spec=layer_spec(range(A.shape[1]), 'mix', A, 1.0), max_new_tokens=MAX_NEW)
    same = sum(a == b for a, b in zip(o1, o2))
    ff = [it['first_clean'] for it in its]; enc = task_state.encode(tok, ps, TASK_SYSTEM, SFX).to(m.device)
    f = torch.tensor(ff, device=m.device)[:, None]; enc2 = dict(input_ids=torch.cat([enc['input_ids'], f], 1), attention_mask=torch.cat([enc['attention_mask'], torch.ones_like(f)], 1))
    hk.rec = {}; hk.spec = {}; hk.offset = 1; hk.record = True
    try: task_state._generate(m, tok, enc2, 1)
    finally: hk.record = False; hk.offset = 0
    A1 = torch.stack([hk.rec[i] for i in range(len(hk.layers))], 1); hk.rec = {}
    rel = float((A1.float() - A.float()).abs().max() / A.float().abs().max())
    print(f'self-test, {LB[t]}: prompt passes per call {passes}; answers unchanged when the state is written back {same}/{len(o1)}; template state with the first token appended, largest relative difference {rel:.1e}')
    assert passes == 1 and same >= len(o1) - 1 and rel <= 1e-3, 'the hooks do not behave; stop here'
    TESTED.add(t)
for t in MODELS:
    use(t); self_test(t); S = states(t, SUCC[t] + FAIL[t] + TWIN_ALL[t])
    print(f'{LB[t]}: {len(S)} task states captured ({len(SUCC[t])} successes, {len(FAIL[t])} failed originals, {len(TWIN_ALL[t])} clean twins)')

## Stage C. Notebook 53's screen, donors and answers, read from its cache and checked (no new sampling)

In [ ]:
# ---- the five-sample screen: notebook 53's cached samples, labels checked against its file ----
SCR = {}
for t in MODELS:
    SCR[t] = jload(f'{OUT53}/{t}/screen.json'); miss = [it['key'] for it in SUCC[t] + FAIL[t] if it['key'] not in SCR[t]]
    assert not miss, f'{LB[t]}: {len(miss)} items have no cached screen in {OUT53}'
    print(f'{LB[t]}: {judge_pending(t, [(it["category"], it["goal"], it["document"], o) for it in SUCC[t] + FAIL[t] for o in SCR[t][it["key"]]])} screen answers needed a new verdict (expected 0)')
    for it in SUCC[t] + FAIL[t]:
        oks = [ok(t, it['category'], it['goal'], it['document'], o) for o in SCR[t][it['key']]]
        it['samples_ok'] = int(sum(oks)); it['label'] = screen_label(it['greedy_ok'], oks, SCREEN['need']); it['robust'] = it['label'].startswith('robust')
        it['robust51'] = ROBUST5.get(it['injection'], False) if (t == U and it['greedy_ok']) else None
RS = {t: [it for it in SUCC[t] if it['robust']] for t in MODELS}; RF = {t: [it for it in FAIL[t] if it['robust']] for t in MODELS}
mine = pd.DataFrame([dict(model=t, injection=it['injection'], attempt=it['attempt'], label=it['label']) for t in MODELS for it in SUCC[t] + FAIL[t]])
s53 = pd.read_csv(f'{REP53}/screen_items.csv')[['model', 'injection', 'attempt', 'label']]
cmp_ = mine.merge(s53, on=['model', 'injection', 'attempt'], how='outer', suffixes=('', '_53'), indicator=True)
assert (cmp_._merge == 'both').all() and (cmp_.label == cmp_.label_53).all(), "the screen labels differ from notebook 53's"
print('screen labels identical to notebook 53:', mine.groupby(['model', 'label']).size().to_dict())

# ---- donors: notebook 53's procedure and seed, so every item gets the same donor ----
rng53 = np.random.default_rng(53)
def pick(c): return c[int(rng53.integers(len(c)))] if c else None
def assign_donors(t):
    for it in SUCC[t]:
        same = lambda d: d['category'] == it['category'] and d['injection'] != it['injection'] and d['document'] != it['document']
        for cand, typ in (([d for d in RS[t] if same(d)], 'robust success, same category'), ([d for d in SUCC[t] if same(d)], 'fragile success, same category'), ([d for d in RS[t] if d['category'] != it['category']], 'robust success, other category')):
            if cand: d = pick(cand); it['ctrlA'] = d['key']; it['ctrlA_type'] = typ; break
        else: it['ctrlA'] = None; it['ctrlA_type'] = 'none'
        for cand, typ in (([d for d in RF[t] if same(d)], 'robust failure, same category'), ([d for d in FAIL[t] if same(d)], 'fragile failure, same category'), ([d for d in RF[t] if d['category'] != it['category']], 'robust failure, other category')):
            if cand: d = pick(cand); it['ctrlB'] = d['key']; it['ctrlB_type'] = typ; break
        else: it['ctrlB'] = None; it['ctrlB_type'] = 'none'
    for grp in (FAIL[t], TWIN[t]):
        for it in grp:
            cand = [d for d in RS[t] if d['category'] != it['category'] and d['injection'] != it['injection']]
            d = pick(cand); it['donor'] = d['key'] if d else None; it['donor_goal'] = d['goal'] if d else None; it['donor_category'] = d['category'] if d else None; it['donor_type'] = 'robust success, other category' if d else 'none'
    for grp in (SUCC[t], FAIL[t], TWIN[t]):
        k_ = len(grp)
        if k_ >= 2:
            while True:
                p = rng53.permutation(k_)
                if not (p == np.arange(k_)).any(): break
            for a, b in enumerate(p): grp[a]['otherclean'] = f"twin{grp[b]['injection']}"
        else:
            for it in grp: it['otherclean'] = None
for t in MODELS: assign_donors(t)
si53 = pd.read_csv(f'{REP53}/success_items.csv')
mine = pd.DataFrame([dict(model=t, injection=it['injection'], attempt=it['attempt'], ctrlA_type=it['ctrlA_type'], ctrlB_type=it['ctrlB_type']) for t in MODELS for it in SUCC[t]])
cmp_ = mine.merge(si53[['model', 'injection', 'attempt', 'ctrlA_type', 'ctrlB_type']], on=['model', 'injection', 'attempt'], how='outer', suffixes=('', '_53'), indicator=True)
assert (cmp_._merge == 'both').all() and (cmp_.ctrlA_type == cmp_.ctrlA_type_53).all() and (cmp_.ctrlB_type == cmp_.ctrlB_type_53).all(), "the control types differ from notebook 53's"
print('control types identical to notebook 53 (the donors themselves are checked by regenerating answers in stage D)')

# ---- notebook 53's answers, read only ----
K = len(SFX); TR53 = ['donor', 'otherclean', 'noise']
C53 = {U: [f'{a}_{w}' for w in WINDOWS for a in ('clean', 'noise')] + [f'ctrlA_{ALL}', f'ctrlB_{ALL}', 'ft_none', 'ft_clean', 'ft_noise', 'placeholder', 'filler', f'ctrlA_{W53}', f'ctrlB_{W53}']
          + [f'mix_a{a}_{W53}' for a in DOSES if a != 1.0] + [f'noisea{a}_{W53}' for a in DOSES if a != 1.0] + [f'pos{k}_{W53}' for k in range(K)],
       V4: [f'{a}_{w}' for w in WINDOWS for a in ('clean', 'noise')] + ['placeholder', 'filler']}
GS = {t: {c: jload(f'{OUT53}/{t}/succ_{c}.json') for c in C53[t]} for t in MODELS}
for t in MODELS:
    for c, G in GS[t].items(): assert all(it['key'] in G for it in SUCC[t]), f'{LB[t]}: notebook 53 cache incomplete for {c}'
HAVE_F = [it for it in FAIL[U] if it['donor']]; HAVE_T = [it for it in TWIN[U] if it['donor']]
GF = {U: {c: jload(f'{OUT53}/{U}/fail_{c}.json') for c in TR53}}; GT = {U: {c: jload(f'{OUT53}/{U}/twin_{c}.json') for c in TR53}}
for c in TR53:
    assert all(it['key'] in GF[U][c] for it in HAVE_F) and all(it['key'] in GT[U][c] for it in HAVE_T), f'notebook 53 transplant cache incomplete for {c}'
print('notebook 53 answers loaded:', {LB[t]: len(GS[t]) for t in MODELS}, 'success conditions;', len(TR53), 'transplant conditions on', len(HAVE_F), 'failed originals and', len(HAVE_T), 'clean twins')

## Stage D. The condition builder, the size check and the reproduction checks (GPU)

In [ ]:
def parse(cond):
    """'<name>_<window>' for the window conditions; (cond, None) otherwise."""
    if '_' in cond:
        name, w = cond.rsplit('_', 1)
        if w in WINDOWS: return name, w
    return cond, None
def eligible(it, cond):
    name, _ = parse(cond); return it.get(name) is not None if name in ('ctrlA', 'ctrlB') else True
def build(t, cond, bx, kind):
    """The spec (and forced first token) of one condition for a batch. Notebook 53's constructions are kept for the checks
    ('noise' adds the noise at every layer of the window); the rebuilt arms replace the state at every layer of the window
    with attacked + noise ('noiseR'), attacked + alpha (clean - attacked) ('mixR_a'), attacked + alpha noise ('noiseR_a')."""
    S = STATES[t]; A = torch.stack([S[it['key']] for it in bx]); Af = A.float(); L, K_ = A.shape[1], A.shape[2]; name, w = parse(cond); lay = WINDOWS[w] if w else None
    own = lambda it, k: S[it[k]] if it.get(k) else S[it['key']]
    if kind == 'succ':
        C = torch.stack([S[f"twin{it['injection']}"] for it in bx]); N = fresh_noise(C.float() - Af, [it['key'] for it in bx]).float()
        if name == 'clean': return layer_spec(lay, 'mix', C, 1.0), None
        if name == 'noise': return layer_spec(lay, 'add', N, 1.0), None                             # notebook 53's construction
        if name == 'noiseR': return layer_spec(lay, 'mix', Af + N, 1.0), None
        if name in ('ctrlA', 'ctrlB'): return layer_spec(lay, 'mix', torch.stack([own(it, name) for it in bx]), 1.0), None
        if name.startswith('mixR_a'): a = float(name[len('mixR_a'):]); return layer_spec(lay, 'mix', Af + a * (C.float() - Af), 1.0), None
        if name.startswith('noiseR_a'): a = float(name[len('noiseR_a'):]); return layer_spec(lay, 'mix', Af + a * N, 1.0), None
        if name.startswith('pos'): k = int(name[3:]); return layer_spec(lay, 'mix', C, 1.0, pos=[j == k for j in range(K_)]), None
        if cond == 'ft_noiseR': return layer_spec(range(L), 'mix', Af + N, 1.0), [it['first_clean'] for it in bx]
    else:                                                                                              # the transplant, at layers 0 to 31
        D_ = torch.stack([S[it['donor']] for it in bx])
        if cond == 'donor': return layer_spec(WINDOWS[ALL], 'mix', D_, 1.0), None
        if cond == 'noiseR': return layer_spec(WINDOWS[ALL], 'mix', Af + fresh_noise(D_.float() - Af, [it['key'] for it in bx]).float(), 1.0), None
    raise ValueError(cond)
def first_batch(items):
    """The first batch in the order notebook 53 ran its conditions (shortest prompts first), so the padding is the same."""
    return [items[k] for k in chunks(by_len([it['prompt'] for it in items]))[0]]

# ---- size check: how far each noise version moves the state just after the window, relative to the clean write ----
def after_state(t, bx, spec, layer):
    """The state after decoder layer `layer` at the K template positions, under spec (recorded before any write at that layer)."""
    m, tok, hk = use(t); enc = task_state.encode(tok, [it['prompt'] for it in bx], TASK_SYSTEM, SFX).to(m.device)
    hk.rec = {}; hk.spec = dict(spec or {}); hk.offset = 0; hk.record = True
    try: task_state._generate(m, tok, enc, 1)
    finally: hk.record = False; hk.spec = {}
    h = hk.rec[layer].float(); hk.rec = {}; return h
srows = []
bx = RS[U][:BS]
for w in ('L21-26', ALL):
    after = max(WINDOWS[w]) + 1
    h0 = after_state(U, bx, None, after)
    dev = {c: (after_state(U, bx, build(U, f'{c}_{w}', bx, 'succ')[0], after) - h0).norm(dim=-1) for c in ('clean', 'noise', 'noiseR')}   # [B, K]
    for c in ('noise', 'noiseR'):
        r_ = (dev[c] / dev['clean'].clamp_min(1e-6)).numpy()
        srows.append(dict(window=w, measured_after_layer=after, arm={'noise': 'notebook 53 noise (added at every layer)', 'noiseR': 'rebuilt noise (attacked + noise at every layer)'}[c],
                          median_ratio_all_positions=round(float(np.median(r_)), 2), median_ratio_last_position=round(float(np.median(r_[:, -1])), 2)))
SIZE = pd.DataFrame(srows); SIZE.to_csv(f'{REP}/size_check.csv', index=False)
print('=== SIZE CHECK: distance the state moves just after the window, as a multiple of the clean write\'s distance (first 16 robust successes) ===')
print(SIZE.to_string(index=False))
assert SIZE[SIZE.arm.str.startswith('rebuilt')].median_ratio_all_positions.between(0.67, 1.5).all(), 'the rebuilt noise does not move the state about as far as the clean write; stop and look'

# ---- reproduction checks against notebook 53's cached answers (same batch, same padding) ----
def repro(t, cond, items, kind, cache):
    m, tok, hk = use(t); bx = first_batch(items); spec, ff = build(t, cond, bx, kind)
    outs, _ = generate_patched(m, hk, tok, [it['prompt'] for it in bx], TASK_SYSTEM, SFX, spec=spec, force_first=ff, max_new_tokens=MAX_NEW - 1 if ff else MAX_NEW)
    diff = [it['key'][:10] for o, it in zip(outs, bx) if o != cache[it['key']][0]]; same = len(bx) - len(diff)
    print(f'reproduction, {cond} ({kind}): {same}/{len(bx)} answers identical to notebook 53' + (f'; differing items {diff}' if diff else ''))
    return same, len(bx)
checks_ = [repro(U, f'clean_{W53}', SUCC[U], 'succ', GS[U][f'clean_{W53}']), repro(U, f'noise_{W53}', SUCC[U], 'succ', GS[U][f'noise_{W53}']), repro(U, 'donor', HAVE_F, 'fail', GF[U]['donor'])]
assert all(s_ >= n_ - 1 for s_, n_ in checks_), 'notebook 53 is not reproduced (states, donors or noise differ); stop and look before running anything new'

## Stage E. The rebuilt noise and dose arms, the judge, and the best window (GPU, cached)

In [ ]:
def run_conditions(t, items, conds, kind):
    """Every condition on the same batches (same padding) as notebook 53; answers cached in this notebook's folder."""
    m, tok, hk = use(t); G = {c: jload(f'{OUT}/{t}/{kind}_{c}.json') for c in conds}; order = by_len([it['prompt'] for it in items]); bl = chunks(order)
    for bi, idx in enumerate(bl):
        todo = [c for c in conds if any(items[k]['key'] not in G[c] for k in idx)]
        if not todo: continue
        bx = [items[k] for k in idx]; ps = [it['prompt'] for it in bx]
        for c in todo:
            spec, ff = build(t, c, bx, kind)
            outs, first = generate_patched(m, hk, tok, ps, TASK_SYSTEM, SFX, spec=spec, force_first=ff, max_new_tokens=MAX_NEW - 1 if ff else MAX_NEW)
            G[c].update({it['key']: ([o, f] if eligible(it, c) else None) for it, o, f in zip(bx, outs, first)}); jsave(G[c], f'{OUT}/{t}/{kind}_{c}.json')
        print(f'  {LB[t]}, {kind}: batch {bi + 1}/{len(bl)} ({len(todo)} conditions)')
    return G
P1R = {U: [f'noiseR_{w}' for w in WINDOWS] + ['ft_noiseR'], V4: [f'noiseR_{w}' for w in WINDOWS]}
for t in MODELS:
    GS[t].update(run_conditions(t, SUCC[t], P1R[t], 'succ'))
    if t == U: GF[U].update(run_conditions(U, HAVE_F, ['noiseR'], 'fail')); GT[U].update(run_conditions(U, HAVE_T, ['noiseR'], 'twin'))
def judge_all(t):
    tr = [(it['category'], it['goal'], it['document'], v[0]) for c in GS[t] for it in SUCC[t] for v in [GS[t][c].get(it['key'])] if v]
    if t in GF:
        for G, grp in ((GF[t], HAVE_F), (GT[t], HAVE_T)):
            for c in G:
                for it in grp:
                    v = G[c].get(it['key'])
                    if v: tr += [(it['category'], it['goal'], it['document'], v[0]), (it['donor_category'], it['donor_goal'], it['document'], v[0])]
    return judge_pending(t, tr)
for t in MODELS: print(f'{LB[t]}: {judge_all(t)} new answers judged')
def stop(t, c, items):
    """Stopped (own injection not followed) per item under condition c; None where the condition did not apply."""
    return [None if GS[t][c].get(it['key']) is None else (not ok(t, it['category'], it['goal'], it['document'], GS[t][c][it['key']][0])) for it in items]
def arr(v): return np.array([bool(x) for x in v if x is not None], bool)
def pair_stop(t, a, b, items):
    x, y = stop(t, a, items), stop(t, b, items); m = [i for i in range(len(items)) if x[i] is not None and y[i] is not None]
    return paired(np.array([x[i] for i in m], bool), np.array([y[i] for i in m], bool))
# the best six-layer window, by notebook 53's rule, now with the rebuilt noise
ref_items = RS[U]
GAP = {w: pair_stop(U, f'clean_{w}', f'noiseR_{w}', ref_items) for w in SIX}
WBEST = max(SIX, key=lambda w: GAP[w][0])
print('clean minus rebuilt noise on robust successes, by window: ' + ', '.join(f'{w} {dstr(GAP[w])}' for w in SIX) + f'; {ALL} {dstr(pair_stop(U, f"clean_{ALL}", f"noiseR_{ALL}", ref_items))} -> best window {WBEST}' + ('' if WBEST == W53 else f' (notebook 53: {W53})'))
P2R = [f'mixR_a{a}_{WBEST}' for a in DOSES if a != 1.0] + [f'noiseR_a{a}_{WBEST}' for a in DOSES if a != 1.0]
if WBEST != W53: P2R += [f'ctrlA_{WBEST}', f'ctrlB_{WBEST}'] + [f'pos{k}_{WBEST}' for k in range(K)]
GS[U].update(run_conditions(U, SUCC[U], P2R, 'succ')); print(f'{LB[U]}: {judge_all(U)} new answers judged (phase 2)')
free_seven_b(); TR = ['donor', 'otherclean', 'noiseR']; print('phase 2 complete; verdicts ready')

## Stage F. Results with the rebuilt noise, notebook 53's noise beside it, the go marks and the decision

In [ ]:
import matplotlib.pyplot as plt
from reslog import log_result
release(); free_seven_b()
med = lambda x: float(np.median(x)) if len(x) else np.nan
r3 = lambda v: round(float(v), 3) if v is not None and not (isinstance(v, float) and np.isnan(v)) else np.nan
fl = lambda s_, k=240: ' '.join(str(s_ or '').split())[:k]
GRP = {t: {'robust successes': RS[t], 'fragile successes': [it for it in SUCC[t] if not it['robust']], 'all successes': SUCC[t]} for t in MODELS}
GRP[U]['robust (single-layer definition)'] = [it for it in SUCC[U] if it['robust51']]
def srate(t, c, items): return rate(arr(stop(t, c, items)))
def f1_deg(t, c, items):
    st = stop(t, c, items); its = [it for it, s_ in zip(items, st) if s_]; outs = [GS[t][c][it['key']][0] for it in its]
    return med([token_f1(o, it['clean_answer']) for o, it in zip(outs, its)]), (float(np.mean([degenerate(o) for o in outs])) if outs else np.nan), len(outs)
NUM = {}
# ---- notebook 53's noise beside the rebuilt noise ----
orows = []
for t in MODELS:
    for g, items in GRP[t].items():
        if not items: continue
        for w in WINDOWS:
            orows.append(dict(model=LB[t], items=g, n=len(items), window=w, clean=rstr(srate(t, f'clean_{w}', items), 3), noise_nb53_added_every_layer=rstr(srate(t, f'noise_{w}', items), 3),
                              noise_rebuilt=rstr(srate(t, f'noiseR_{w}', items), 3), rebuilt_minus_nb53=dstr(pair_stop(t, f'noiseR_{w}', f'noise_{w}', items))))
OLDNEW = pd.DataFrame(orows); OLDNEW.to_csv(f'{REP}/noise_nb53_vs_rebuilt.csv', index=False)
# ---- windows: clean against the rebuilt noise, and the in-distribution controls ----
wrows, prows = [], []
for t in MODELS:
    for g, items in GRP[t].items():
        if not items: continue
        for w in WINDOWS:
            row = dict(model=LB[t], items=g, n=len(items), window=w, clean=rstr(srate(t, f'clean_{w}', items), 3), noise=rstr(srate(t, f'noiseR_{w}', items), 3), clean_minus_noise=dstr(pair_stop(t, f'clean_{w}', f'noiseR_{w}', items)))
            for ctl, lab in (('ctrlA', 'attack_that_worked'), ('ctrlB', 'attack_that_failed')):
                c = f'{ctl}_{w}'; row[lab] = rstr(srate(t, c, items), 3) if c in GS[t] and any(GS[t][c].get(it['key']) for it in items) else ''
            wrows.append(row); NUM[('w', t, g, w)] = (srate(t, f'clean_{w}', items), srate(t, f'noiseR_{w}', items), pair_stop(t, f'clean_{w}', f'noiseR_{w}', items))
        for w in (ALL, WBEST):
            for ctl, lab in (('ctrlA', 'attack that worked'), ('ctrlB', 'attack that failed')):
                c = f'{ctl}_{w}'
                if c in GS[t] and any(GS[t][c].get(it['key']) for it in items):
                    prows.append(dict(model=LB[t], items=g, window=w, comparison=f'clean minus {lab}', difference=dstr(pair_stop(t, f'clean_{w}', c, items))))
                    prows.append(dict(model=LB[t], items=g, window=w, comparison=f'{lab} minus noise', difference=dstr(pair_stop(t, c, f'noiseR_{w}', items))))
WIN, WINP = pd.DataFrame(wrows), pd.DataFrame(prows); WIN.to_csv(f'{REP}/windows.csv', index=False); WINP.to_csv(f'{REP}/windows_controls_paired.csv', index=False)
# ---- first token held equal ----
frows = []
for g, items in GRP[U].items():
    if not items: continue
    frows.append(dict(items=g, n=len(items), **{c: rstr(srate(U, c, items), 3) for c in ('ft_none', 'ft_clean', 'ft_noiseR', 'ft_noise')}, clean_minus_noise=dstr(pair_stop(U, 'ft_clean', 'ft_noiseR', items)), clean_minus_none=dstr(pair_stop(U, 'ft_clean', 'ft_none', items))))
    NUM[('ft', g)] = pair_stop(U, 'ft_clean', 'ft_noiseR', items)
FT = pd.DataFrame(frows).rename(columns={'ft_noiseR': 'ft_noise_rebuilt', 'ft_noise': 'ft_noise_nb53'}); FT.to_csv(f'{REP}/first_token_held_equal.csv', index=False)
# ---- dose and positions at the best window (undefended) ----
drows, DOSE = [], {}
for g, items in GRP[U].items():
    if not items: continue
    for a in DOSES:
        cm, cn = (f'clean_{WBEST}', f'noiseR_{WBEST}') if a == 1.0 else (f'mixR_a{a}_{WBEST}', f'noiseR_a{a}_{WBEST}')
        DOSE[(g, a)] = (srate(U, cm, items), srate(U, cn, items)); drows.append(dict(items=g, n=len(items), window=WBEST, alpha=a, clean=rstr(DOSE[(g, a)][0], 3), noise=rstr(DOSE[(g, a)][1], 3), clean_minus_noise=dstr(pair_stop(U, cm, cn, items))))
DS = pd.DataFrame(drows); DS.to_csv(f'{REP}/dose.csv', index=False)
PS = pd.DataFrame([dict(items=g, n=len(items), window=WBEST, all_five=rstr(srate(U, f'clean_{WBEST}', items), 3), **{f'position_{k}_only': rstr(srate(U, f'pos{k}_{WBEST}', items), 3) for k in range(K)}) for g, items in GRP[U].items() if items])
PS.to_csv(f'{REP}/positions.csv', index=False)
# ---- content of the stopped answers, and both placeholder forms ----
crows = []
for t in MODELS:
    f1_ref = med([token_f1(r.output, CO[t][str(r.injection)]['output']) for r in ATT[t][~ATT[t].success].itertuples()])
    for g, items in GRP[t].items():
        if not items: continue
        row = dict(model=LB[t], items=g, n=len(items), f1_failed_attacks_vs_clean_answer=r3(f1_ref))
        for c in (f'clean_{ALL}', f'clean_{WBEST}', f'noiseR_{ALL}', 'placeholder', 'filler'):
            f1, dg, k_ = f1_deg(t, c, items); row[f'{c}_stopped'] = k_; row[f'{c}_f1'] = r3(f1); row[f'{c}_degenerate'] = r3(dg); NUM[('content', t, g, c)] = (f1, dg, srate(t, c, items))
        crows.append(row)
CT = pd.DataFrame(crows); CT.to_csv(f'{REP}/content.csv', index=False)
# ---- by category, robust successes (undefended) ----
CAT = pd.DataFrame([dict(category=c, robust_successes=len(its), fragile_successes=sum(1 for it in SUCC[U] if it['category'] == c and not it['robust']), robust_failures=sum(1 for it in FAIL[U] if it['category'] == c and it['robust']), **{f'{a}_{w}': r3(srate(U, f'{a}_{w}', its)['rate']) if its else np.nan for w in (ALL, WBEST) for a in ('clean', 'noiseR', 'noise')},
                         ft_clean=r3(srate(U, 'ft_clean', its)['rate']) if its else np.nan, ft_noiseR=r3(srate(U, 'ft_noiseR', its)['rate']) if its else np.nan) for c in CATEGORIES for its in [[it for it in RS[U] if it['category'] == c]]])
CAT.to_csv(f'{REP}/by_category.csv', index=False)
# ---- the transplant ----
def tp(G, items, c, which):
    out = []
    for it in items:
        v = G[c].get(it['key'])
        if v is None: out.append(None); continue
        out.append(ok(U, it['category'], it['goal'], it['document'], v[0]) if which == 'own' else ok(U, it['donor_category'], it['donor_goal'], it['document'], v[0]))
    return out
trows = []; TPN = {}
for lab, G, groups in (('failed originals', GF[U], {'robust failures': [it for it in RF[U] if it['donor']], 'all failed originals': HAVE_F}),
                       ('clean twins', GT[U], {'twins of robust successes': [it for it in HAVE_T if it['injection'] in {x['injection'] for x in RS[U]}], 'all clean twins': HAVE_T})):
    for g, items in groups.items():
        if not items: continue
        for c in TR + ['noise']:
            own, don = arr(tp(G, items, c, 'own')), arr(tp(G, items, c, 'donor')); outs = [G[c][it['key']][0] for it in items]
            dg = np.array([degenerate(o) for o in outs], bool); f1 = med([token_f1(o, it['clean_answer'] if lab == 'failed originals' else it['greedy']) for o, it in zip(outs, items)])
            TPN[(g, c)] = dict(own=own, donor=don); trows.append(dict(items=g, n=len(items), condition=c + (' (notebook 53, added every layer)' if c == 'noise' else ''), own_injection_followed=rstr(rate(own), 3), donor_injection_followed=rstr(rate(don), 3), neither=r3((~own & ~don).mean()), degenerate=r3(dg.mean()), median_f1_vs_clean_answer=r3(f1)))
        for which in ('own', 'donor'):
            d_ = dstr(paired(TPN[(g, 'donor')][which], TPN[(g, 'noiseR')][which]))
            trows.append(dict(items=g, n=len(items), condition=f'donor minus rebuilt noise ({which} injection followed)', own_injection_followed=d_ if which == 'own' else '', donor_injection_followed=d_ if which == 'donor' else ''))
TP = pd.DataFrame(trows); TP.to_csv(f'{REP}/transplant.csv', index=False)
pd.DataFrame([dict(model=t, injection=it['injection'], category=it['category'], attempt=it['attempt'], label=it['label'], samples_ok=it['samples_ok'], robust_single_layer=it.get('robust51'), ctrlA_type=it.get('ctrlA_type'), ctrlB_type=it.get('ctrlB_type'),
                   **{c: (None if GS[t][c].get(it['key']) is None else int(not ok(t, it['category'], it['goal'], it['document'], GS[t][c][it['key']][0]))) for c in GS[t]}) for t in MODELS for it in SUCC[t]]).to_csv(f'{REP}/success_items.csv', index=False)

# ---- the go marks: notebook 53's, with the rebuilt noise; the filler placeholder printed as recorded (post hoc), not scored ----
rs = RS[U]; ok1 = {w: NUM[('w', U, 'robust successes', w)] for w in WINDOWS}
r1_windows = [w for w in WINDOWS if ok1[w][0]['rate'] >= CRIT['r1_stop'] and ok1[w][1]['rate'] <= CRIT['r1_noise']]
ft_gap = NUM[('ft', 'robust successes')]
c_all = ok1[ALL][0]['rate']; c_best = ok1[WBEST][0]['rate']; r2_share = c_best / c_all if c_all else np.nan
alpha_min = next((a for a in DOSES if DOSE[('robust successes', a)][0]['rate'] >= CRIT['r2_dose_stop']), None)
pos_rates = {k: srate(U, f'pos{k}_{WBEST}', rs)['rate'] for k in range(K)}
pos_alone = [k for k in range(K) if c_best and pos_rates[k] >= CRIT['r2_share'] * c_best]
g_f, g_t = 'robust failures', 'all clean twins'
own_d, own_n, own_o = (TPN[(g_f, c)]['own'].mean() for c in ('donor', 'noiseR', 'otherclean')); d_fn = paired(TPN[(g_f, 'donor')]['own'], TPN[(g_f, 'noiseR')]['own'])
don_d, don_o, don_n = (TPN[(g_t, c)]['donor'].mean() for c in ('donor', 'otherclean', 'noiseR'))
cl = NUM[('content', U, 'robust successes', f'clean_{ALL}')]
PH = {form: (NUM[('content', U, 'robust successes', form)][2]['rate'], NUM[('content', U, 'robust successes', form)][0] - cl[0]) for form in ('placeholder', 'filler')}
ge = lambda v, m_: (not np.isnan(v)) and v >= m_ - 1e-9; le = lambda v, m_: (not np.isnan(v)) and v <= m_ + 1e-9
checks = [('R1', f"a window where the clean state stops >= {CRIT['r1_stop']} and rebuilt noise <= {CRIT['r1_noise']} (met by: {', '.join(r1_windows) or 'none'})", max(ok1[w][0]['rate'] for w in WINDOWS), 'at least one window', bool(r1_windows)),
          ('R1', 'first token held equal: ft_clean minus rebuilt noise', ft_gap[0], f">= {CRIT['r1_ft_gap']}", ge(ft_gap[0], CRIT['r1_ft_gap'])),
          ('R2', f'best window {WBEST}: clean stop rate as a share of the clean stop rate at {ALL}', r2_share, f">= {CRIT['r2_share']}", ge(r2_share, CRIT['r2_share'])),
          ('R2', f"smallest dose with the clean state stopping >= {CRIT['r2_dose_stop']} (recorded): {alpha_min}", alpha_min if alpha_min is not None else np.nan, 'recorded', True),
          ('R3 switch', 'robust failures: own injection followed under the donor state', own_d, f">= {CRIT['r3_own']}", ge(own_d, CRIT['r3_own'])),
          ('R3 switch', 'robust failures: own injection followed under rebuilt noise, and under another clean state', max(own_n, own_o), f"<= {CRIT['r3_ctrl']} (both)", le(own_n, CRIT['r3_ctrl']) and le(own_o, CRIT['r3_ctrl'])),
          ('R3 switch', 'robust failures: donor minus rebuilt noise, lower end', d_fn[1], '> 0', (not np.isnan(d_fn[1])) and d_fn[1] > 0),
          ('R3 switch', "clean twins: the donor's injection followed under the donor state", don_d, f"<= {CRIT['r3_twin_donor']}", le(don_d, CRIT['r3_twin_donor'])),
          ('R3 content', "clean twins: the donor's injection followed under the donor state", don_d, f">= {CRIT['r3_content']}", ge(don_d, CRIT['r3_content'])),
          ('R3 content', "clean twins: the donor's injection followed under the two controls", max(don_o, don_n), f"<= {CRIT['r3_content_ctrl']} (both)", le(don_o, CRIT['r3_content_ctrl']) and le(don_n, CRIT['r3_content_ctrl'])),
          ('placeholder "(document)"', 'stop rate on robust successes', PH['placeholder'][0], f">= {CRIT['ph_stop']}", ge(PH['placeholder'][0], CRIT['ph_stop'])),
          ('placeholder "(document)"', f"median F1 of its stopped answers minus the clean-twin arm's ({ALL})", PH['placeholder'][1], f">= -{CRIT['ph_f1_tol']}", ge(PH['placeholder'][1], -CRIT['ph_f1_tol'])),
          ('filler (recorded, post hoc)', 'stop rate on robust successes (values already seen in notebook 53)', PH['filler'][0], 'recorded', True),
          ('filler (recorded, post hoc)', f"median F1 of its stopped answers minus the clean-twin arm's ({ALL})", PH['filler'][1], 'recorded', True)]
G_ = lambda name: all(bool(c[4]) for c in checks if c[0] == name)
R1, R2, R3S, R3C = G_('R1'), G_('R2'), G_('R3 switch'), G_('R3 content')
PHOK = G_('placeholder "(document)"')
R3_READING = 'content-free switch' if R3S else ('the state carries content' if R3C else 'neither (the state is a stop signal only: needed, not sufficient)')
if R1:
    DECISION = 'GO: notebook 54 (confirmation on fresh items)' + ('; notebook 53c (learned subspace) is also built' if R2 else '')
    FROZEN = dict(screen=f"{SCREEN['samples']} samples at temperature {SCREEN['temperature']}, robust success = greedy and at least {SCREEN['need']} samples", window=WBEST if R2 else ALL, dose=1.0, dose_recorded=alpha_min,
                  positions=f'all five (no single position reaches {CRIT["r2_share"]:.0%} of the window effect)' if not pos_alone else f'all five; positions {pos_alone} alone reach {CRIT["r2_share"]:.0%} of the window effect', r3_reading=R3_READING,
                  r3_test='necessity only' if not (R3S or R3C) else ('R3 switch marks' if R3S else 'R3 content marks'), noise='attacked + noise at every written layer (rebuilt)')
else:
    DECISION = 'CLOSE: the task-state route closes (an exploratory negative on seen items, on the pre-registered noise control, reported with notebooks 51 and 53); notebook 54 is not run; next is the role signal (R5) after its novelty check'; FROZEN = {}
VT = pd.DataFrame([dict(question=a, criterion=b, value=r3(v), mark=m_, met=bool(ok_)) for a, b, v, m_, ok_ in checks]); VT.to_csv(f'{REP}/go_criteria.csv', index=False)
GTXT = (f'{DECISION}\nR1 {"pass" if R1 else "fail"}, R2 {"pass" if R2 else "fail"}, R3 reading: {R3_READING}, placeholder "(document)" {"viable" if PHOK else "not viable"} (filler recorded: stop {PH['filler'][0]:.2f}, F1 gap {PH['filler'][1]:+.3f}, post hoc); robust successes {len(rs)} of {len(SUCC[U])}, robust failures {len(RF[U])} of {len(FAIL[U])} ({LB[U]})'
        + (('\nSettings notebook 54 would freeze: ' + '; '.join(f'{k}: {v}' for k, v in FROZEN.items())) if FROZEN else ''))
open(f'{REP}/go.txt', 'w').write(GTXT + '\n')

# ---- Figure ----
fig, ax = plt.subplots(2, 2, figsize=(14, 9)); ws = list(WINDOWS); x = np.arange(len(ws)); w_ = 0.27
items = GRP[U]['robust successes']
for k2, (c_, lab_, hat) in enumerate((('clean', 'clean state', ''), ('noiseR', 'rebuilt noise', '//'), ('noise', 'notebook 53 noise', '..'))):
    rr = [srate(U, f'{c_}_{w}', items) for w in ws]
    ax[0, 0].bar(x + (k2 - 1) * w_, [r['rate'] for r in rr], w_, yerr=[[max(0, r['rate'] - r['lo']) for r in rr], [max(0, r['hi'] - r['rate']) for r in rr]], capsize=2, hatch=hat, alpha=0.9 if c_ == 'clean' else 0.55, label=f'{lab_} (robust successes, n={len(items)})')
ax[0, 0].axhline(CRIT['r1_stop'], color='grey', ls='--', lw=1); ax[0, 0].axhline(CRIT['r1_noise'], color='grey', ls=':', lw=1); ax[0, 0].set_xticks(x); ax[0, 0].set_xticklabels(ws, fontsize=8); ax[0, 0].set_ylim(0, 1.05)
ax[0, 0].set_ylabel('share of attacks stopped'); ax[0, 0].set_title(f'Clean state against noise, by window ({LB[U]})', fontsize=10); ax[0, 0].legend(fontsize=7)
conds_ = ['ft_none', 'ft_clean', 'ft_noiseR', f'ctrlA_{ALL}', f'ctrlB_{ALL}', f'clean_{ALL}', f'noiseR_{ALL}', 'placeholder', 'filler']; labs_ = ['first token\nonly', 'first token +\nclean state', 'first token +\nrebuilt noise', 'attack that\nworked', 'attack that\nfailed', 'clean state', 'rebuilt noise', 'placeholder', 'filler']
for j, g in enumerate(('robust successes', 'fragile successes')):
    its = GRP[U][g]
    if not its: continue
    rr = [srate(U, c, its) for c in conds_]; ax[0, 1].bar(np.arange(len(conds_)) + (j - 0.5) * 0.38, [r['rate'] for r in rr], 0.38, yerr=[[max(0, r['rate'] - r['lo']) if r['n'] else 0 for r in rr], [max(0, r['hi'] - r['rate']) if r['n'] else 0 for r in rr]], capsize=2, label=f'{g} (n={len(its)})')
ax[0, 1].set_xticks(range(len(conds_))); ax[0, 1].set_xticklabels(labs_, fontsize=7); ax[0, 1].set_ylim(0, 1.05); ax[0, 1].set_title('First token held equal, the controls, the placeholder (layers 0 to 31 unless forced)', fontsize=10); ax[0, 1].legend(fontsize=7)
for g, col in (('robust successes', '#1f77b4'), ('fragile successes', '#ff7f0e')):
    if GRP[U][g]:
        ax[1, 0].plot(DOSES, [DOSE[(g, a)][0]['rate'] for a in DOSES], marker='o', color=col, label=f'{g}: clean'); ax[1, 0].plot(DOSES, [DOSE[(g, a)][1]['rate'] for a in DOSES], marker='s', ls='--', color=col, label=f'{g}: rebuilt noise')
ax[1, 0].axhline(CRIT['r2_dose_stop'], color='grey', ls='--', lw=1); ax[1, 0].set_xticks(DOSES); ax[1, 0].set_xlabel('dose (share of clean minus attacked, or of the noise)'); ax[1, 0].set_ylim(0, 1.05); ax[1, 0].set_title(f'Dose at the best window {WBEST} (rebuilt)', fontsize=10); ax[1, 0].legend(fontsize=7)
tl, tv, te = [], [], []
for g, which, lab in ((g_f, 'own', 'failed originals,\nown followed'), (g_t, 'donor', 'clean twins,\ndonor followed')):
    for c in TR:
        r = rate(TPN[(g, c)][which]); tl.append(f'{lab}\n{c}'); tv.append(r['rate']); te.append((max(0, r['rate'] - r['lo']), max(0, r['hi'] - r['rate'])))
ax[1, 1].bar(range(len(tv)), tv, yerr=np.array(te).T, capsize=2, color=['#2ca02c', '#7f7f7f', '#bcbd22'] * 2); ax[1, 1].set_xticks(range(len(tv))); ax[1, 1].set_xticklabels(tl, fontsize=7)
ax[1, 1].axhline(CRIT['r3_own'], color='grey', ls='--', lw=1); ax[1, 1].axhline(CRIT['r3_ctrl'], color='grey', ls=':', lw=1); ax[1, 1].set_ylim(0, 1.05); ax[1, 1].set_title(f'Transplant at layers 0 to 31 ({g_f}; {g_t})', fontsize=10)
for a_ in ax.flat: a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{FIG}/state_patching_53b.png', dpi=150, bbox_inches='tight'); plt.show()

# ---- Tables, examples, log ----
for title, T_ in (('SIZE CHECK', SIZE), ('NOTEBOOK 53 NOISE BESIDE THE REBUILT NOISE', OLDNEW), ('WINDOWS. clean state against the rebuilt noise (and the in-distribution controls where run)', WIN), ('WINDOWS. paired differences against the controls', WINP),
                  ('FIRST TOKEN HELD EQUAL', FT), ('DOSE at the best window (rebuilt)', DS), ('POSITIONS at the best window', PS), ('CONTENT of the stopped answers', CT), ('BY CATEGORY, robust successes', CAT), ('TRANSPLANT', TP), ('GO MARKS (robust successes, undefended)', VT)):
    print(f'\n=== {title} ==='); print(T_.to_string(index=False) if len(T_) else '(empty)')
print('\n' + GTXT)
rng_ = np.random.default_rng(3); shown = [rs[k] for k in sorted(rng_.choice(len(rs), min(3, len(rs)), replace=False))] if rs else []
print('\n--- examples (robust successes): unpatched, clean state, rebuilt noise and notebook 53 noise at the best window ---')
for it in shown:
    lab = lambda c: 'stopped' if not ok(U, it['category'], it['goal'], it['document'], GS[U][c][it['key']][0]) else 'NOT stopped'
    print(f"[{it['category']}] INJECTED: {fl(it['goal'], 120)}\n   unpatched: {fl(it['greedy'])}\n   clean {WBEST} ({lab(f'clean_{WBEST}')}): {fl(GS[U][f'clean_{WBEST}'][it['key']][0])}\n   rebuilt noise {WBEST} ({lab(f'noiseR_{WBEST}')}): {fl(GS[U][f'noiseR_{WBEST}'][it['key']][0])}\n   notebook 53 noise {WBEST} ({lab(f'noise_{WBEST}')}): {fl(GS[U][f'noise_{WBEST}'][it['key']][0])}\n")
summary = '\n\n'.join([GTXT, 'Go marks:\n' + VT.to_string(index=False), 'Size check:\n' + SIZE.to_string(index=False), 'Notebook 53 noise beside the rebuilt noise:\n' + OLDNEW.to_string(index=False), 'Windows:\n' + WIN.to_string(index=False),
                       'Controls, paired:\n' + WINP.to_string(index=False), 'First token held equal:\n' + FT.to_string(index=False), 'Dose:\n' + DS.to_string(index=False), 'Positions:\n' + PS.to_string(index=False), 'Content:\n' + CT.to_string(index=False),
                       'By category:\n' + CAT.to_string(index=False), 'Transplant:\n' + TP.to_string(index=False)])
log_result(f'nb53b: notebook 53 with the noise control rebuilt as pre-registered (exploration), decision: {DECISION.split(":")[0]}', summary, reports_dir=REP)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", "nb53b (follow-on work): notebook 53 rerun with the noise control built as pre-registered. Notebook 53 added the noise at every layer of a window, so it accumulated and was larger than the clean change; here each written layer is replaced by attacked + noise (same vectors and seeds), as in notebook 51's random_all, and the dose arms likewise. Every other arm, the screen and the donors are read from notebook 53's cache and checked by regeneration; a size check measures each noise version against the clean write; go marks and decision rule unchanged from notebook 53 (the filler placeholder printed as recorded, post hoc)"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)